In [1]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config_domainnet.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config_domainnet.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.data.dataset import IndexedSubset
from ebm_unlearning.src.losses.clip_subspace import (
    compute_domainnet_subspace_weights,
    load_dino_encoder,
    WeightedSubset,
)
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.training.unlearn import UnlearnConfig, unlearn
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker

with open(ROOT / "configs" / "config_domainnet.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))
device = torch.device(cfg.get("device", "cpu"))

from datetime import datetime
logger  = setup_logger("unlearn", log_file=str(ROOT / "outputs" / "logs" / "unlearn_domainnet.log"))
run_id  = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / "domainnet" / "unlearn" / run_id),
)

# Load full DomainNet subset (10 classes × 4 domains)
dset = DomainNetSubset(
    root=str(ROOT / cfg["data"]["data_dir"]),
    classes=cfg["data"]["classes"],
    domains=cfg["data"]["domains"],
)

# Forget: sketch-tiger only | Retain: everything else (incl. real/clipart/painting tiger)
forget_spec = ForgetSpec(
    mode="class_domain",
    class_label=int(cfg["data"]["forget"]["class_label"]),
    domain=str(cfg["data"]["forget"]["domain"]),
)
retain_spec = RetainSpec()
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size  = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

forget_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=True)
# retain_loader rebuilt below after DINO weights are computed

forget_name = cfg["data"]["forget"]["class_name"]
forget_domain = cfg["data"]["forget"]["domain"]
print(f"Forget : {forget_name} ({forget_domain}) — {len(forget_train)} train samples")
print(f"Retain : {len(retain_train)} train samples (includes real/clipart/painting {forget_name})")

2026-06-29 18:06:36.475129: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 18:06:36.529678: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-29 18:06:38.247814: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[domainnet] 13262 images | 10 classes × 4 domains
Forget : tiger (sketch) — 309 train samples
Retain : 10301 train samples (includes real/clipart/painting tiger)


In [3]:
# ── CHANGE THESE only when encoder / forget class / k changes ─────────────────
ENCODER_BACKEND = "dino"   # "clip" | "dino"
# ─────────────────────────────────────────────────────────────────────────────

# Load E0 (pretrained reference — frozen)
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "resnet18")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 2)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

# Load DINOv2 encoder (frozen — only for subspace weight computation)
print(f"Loading {ENCODER_BACKEND.upper()} encoder...")
enc_model, enc_preprocess, enc_type = load_dino_encoder(device)
print(f"{ENCODER_BACKEND.upper()} loaded.")

# Build DINO-preprocessed loaders for feature extraction
from ebm_unlearning.src.data.domainnet import DomainNetSubset
dset_dino = DomainNetSubset(
    root=str(ROOT / cfg["data"]["data_dir"]),
    classes=cfg["data"]["classes"],
    domains=cfg["data"]["domains"],
    transform=enc_preprocess,   # DINOv2 preprocessing
)

from ebm_unlearning.src.data.split import split_forget_retain as sfr
forget_dino_all, retain_dino_all = sfr(dset_dino, forget_spec, retain_spec)
forget_dino_train, _ = train_holdout_split(forget_dino_all, holdout_fraction, seed=int(cfg["seed"]))
retain_dino_train, _ = train_holdout_split(retain_dino_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

forget_dino_loader = DataLoader(forget_dino_train, batch_size=batch_size, shuffle=False, num_workers=0)
retain_dino_loader = DataLoader(retain_dino_train, batch_size=batch_size, shuffle=False, num_workers=0)

n_components = int(cfg["unlearning"].get("n_pca_components", 20))
print(f"\nComputing DINOv2 PCA subspace weights (k={n_components})...")
print(f"Forget subspace computed from: {forget_name} ({forget_domain}) only")

_retain_weights_cache, _, _ = compute_domainnet_subspace_weights(
    model=enc_model,
    forget_loader=forget_dino_loader,
    retain_loader=retain_dino_loader,
    device=device,
    n_components=n_components,
    encoder_type=enc_type,
)

# Zero out CLIP weights for non-forget-class retain samples.
# DINOv2 captures visual style as well as semantics: clipart images (stylized)
# project onto the tiger/sketch subspace due to style similarity, not because
# they are tigers. Applying a forget signal to bear/clipart or lion/clipart
# degrades their accuracy. Only tiger images in other domains should receive
# the cross-domain forget signal.
_forget_cls = int(cfg["data"]["forget"]["class_label"])
_retain_cls = retain_dino_train.base.targets[retain_dino_train.indices]
_retain_weights_cache = _retain_weights_cache.clone()
# _retain_weights_cache[_retain_cls != _forget_cls] = 0.0   # uncomment to restrict to tiger-only (cross-domain mode)
_n_tiger = int((_retain_cls == _forget_cls).sum())
print(f"  cross-class mode: all retain samples receive weighted forget signal")
print(f"  (tiger-only samples: {_n_tiger})")
print("Weights cached. Run next cell to configure lambda and train.")

Loading DINO encoder...


/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DINO loaded.
[domainnet] 13262 images | 10 classes × 4 domains

Computing DINOv2 PCA subspace weights (k=10)...
Forget subspace computed from: tiger (sketch) only
  DINO: extracting forget features (309 samples)...
  DINO: extracting retain features (10301 samples)...
  weights — mean=0.1644  max=0.8499  %>0.05: 69.5%
  cross-class mode: all retain samples receive weighted forget signal
  (tiger-only samples: 1083)
Weights cached. Run next cell to configure lambda and train.


In [4]:
# Diagnostic: per-class average DINO weight
import numpy as np
sorted_classes = sorted(cfg["data"]["classes"])  # alphabetical
print("\n  Per-class avg DINO weight:")
for cls_idx, cls_name in enumerate(sorted_classes):
    mask = (_retain_cls == cls_idx)
    if mask.sum() > 0:
        avg_w = _retain_weights_cache[mask].mean().item()
        print(f"    {cls_name:12s} (idx={cls_idx})  avg_weight={avg_w:.4f}  n={mask.sum().item()}")



  Per-class avg DINO weight:
    airplane     (idx=0)  avg_weight=0.0479  n=681
    bear         (idx=1)  avg_weight=0.1144  n=1001
    car          (idx=2)  avg_weight=0.0419  n=667
    dog          (idx=3)  avg_weight=0.0855  n=1528
    guitar       (idx=4)  avg_weight=0.0556  n=896
    horse        (idx=5)  avg_weight=0.0724  n=1175
    lion         (idx=6)  avg_weight=0.2753  n=1123
    tiger        (idx=7)  avg_weight=0.5703  n=1083
    truck        (idx=8)  avg_weight=0.0478  n=962
    zebra        (idx=9)  avg_weight=0.2365  n=1185


In [5]:
# ── Run this cell whenever lambda_clip / steps changes ────────────────────────
with open(ROOT / "configs" / "config_domainnet.yaml") as f:
    cfg = yaml.safe_load(f)
    
checkpoint_path = str(ROOT / cfg["unlearning"]["checkpoint_path"])

un_cfg = UnlearnConfig(
    steps=int(cfg["unlearning"]["steps"]),
    lr=float(cfg["unlearning"]["lr"]),
    weight_decay=float(cfg["unlearning"]["weight_decay"]),
    lambda_f=float(cfg["unlearning"]["lambda_f"]),
    lambda_r=float(cfg["unlearning"]["lambda_r"]),
    lambda_m=float(cfg["unlearning"]["lambda_m"]),
    lambda_e=float(cfg["unlearning"]["lambda_e"]),
    lambda_clip=float(cfg["unlearning"].get("lambda_clip", 0.0)),
    n_pca_components=int(cfg["unlearning"].get("n_pca_components", 20)),
    margin=float(cfg["unlearning"]["margin"]),
    log_every=int(cfg["unlearning"]["log_every"]),
    checkpoint_path=checkpoint_path,
)

# Reset E to fresh trainable copy
E = deepcopy(E0)
E.train()
for p in E.parameters():
    p.requires_grad_(True)
if hasattr(E, '_backbone') and E._backbone is not None:
    E._set_resnet_trainable_stages(int(cfg["model"].get("finetune_stages", 2)))  # trainable backbone
   # E._set_resnet_trainable_stages(0)  # freeze backbone experiment — made forgetting worse

# Rebuild retain loader with DINO weights
retain_train_weighted = WeightedSubset(retain_train, _retain_weights_cache)
retain_loader = DataLoader(retain_train_weighted, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)

trainable = sum(p.numel() for p in E.parameters() if p.requires_grad)
print(f"lambda_clip={un_cfg.lambda_clip}  steps={un_cfg.steps}  k={un_cfg.n_pca_components}")
print(f"Trainable params: {trainable:,}")
print(f"Checkpoint → {checkpoint_path}")

lambda_clip=1.5  steps=500  k=10
Trainable params: 8,460,801
Checkpoint → /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_unlearned_domainnet_tiger_sketch.pt


In [6]:
E = unlearn(
    E,
    E0,
    forget_loader,
    retain_loader,
    device=device,
    cfg=un_cfg,
    logger=logger,
    tracker=tracker,
    seed=int(cfg["seed"]),
)
tracker.close()

/home/owais/machine unlearning/ebm_unlearning/src/training/unlearn.py:178: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  step, float(gap_fw), float(total),
[2026-06-29 18:08:45,794] [INFO] [unlearn] step=0 gap_fw=-0.3455 total=11.145464 forget=6.309921 retain=0.060935 margin=2.392643 energy_reg=5.646401 clip=1.015486
[2026-06-29 18:09:43,225] [INFO] [unlearn] step=50 gap_fw=4.3006 total=7.342484 forget=1.872404 retain=0.209327 margin=0.442852 energy_reg=13.533560 clip=1.249194
[2026-06-29 18:10:38,728] [INFO] [unlearn] step=100 gap_fw=6.0515 total=4.034137 forget=0.643224 retain=0.111720 margin=0.093425 energy_reg=20.500744 clip=1.067462
[2026-06-29 18:11:33,451] [INFO] [unlearn] step=150 gap_fw=7.4940 total=4.342764 forget=0.206616 retain=0.142363 margin=0.067053 energy_reg=24.775360 clip=1.272

In [8]:
from __future__ import annotations
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION — Per-Class Per-Domain Accuracy
#  Adjust the variables below and run. No other cells needed.
# ══════════════════════════════════════════════════════════════════════════════
UNLEARNED_CHECKPOINT = "outputs/checkpoints/ebm_unlearned_domainnet_tiger_sketch.pt"

# Classes to show — empty list = all 10 classes
FOCUS_CLASSES = ["tiger", "lion", "bear", "zebra", "dog", "truck", "guitar"]

# Domains to show — empty list = all 4 domains
FOCUS_DOMAINS = ["real", "sketch", "clipart", "painting"]
# ══════════════════════════════════════════════════════════════════════════════

import sys
import numpy as np
from pathlib import Path
import torch
import yaml
from torch.utils.data import DataLoader

def _find_root():
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "configs" / "config_domainnet.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("project root not found")

ROOT = _find_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.evaluation.classification import predict_argmin_energy

with open(ROOT / "configs" / "config_domainnet.yaml") as f:
    cfg = yaml.safe_load(f)

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = int(cfg["model"].get("num_classes", 10))
dn_root     = str(ROOT / cfg["data"]["data_dir"])
classes     = cfg["data"]["classes"]
forget_name = cfg["data"]["forget"]["class_name"]
forget_dom  = cfg["data"]["forget"]["domain"]

def _make_model():
    return EnergyModel(
        in_channels=int(cfg["model"]["in_channels"]),
        hidden_dim=int(cfg["model"]["hidden_dim"]),
        num_classes=num_classes,
        embed_dim=int(cfg["model"].get("embed_dim", 128)),
        backbone=str(cfg["model"].get("backbone", "resnet18")),
        finetune_stages=int(cfg["model"].get("finetune_stages", 2)),
        imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
    )

print("Loading models...")
E0 = load_pretrained(_make_model(), str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)
E  = load_pretrained(_make_model(), str(ROOT / UNLEARNED_CHECKPOINT), device=device)
E0.eval(); E.eval()
print(f"  Pretrained : {cfg['pretrain']['checkpoint_path']}")
print(f"  Unlearned  : {UNLEARNED_CHECKPOINT}")

# ── Per-domain per-class evaluation ──────────────────────────────────────────
def eval_per_domain_class(model, domains):
    results = {}
    for domain in domains:
        dset_d  = DomainNetSubset(root=dn_root, classes=classes, domains=[domain])
        loader  = DataLoader(dset_d, batch_size=32, shuffle=False, num_workers=0)
        yt, yp  = predict_argmin_energy(model, loader, device=device, num_classes=num_classes, y_chunk=5)
        for idx, cls in enumerate(dset_d.classes):
            mask = yt == idx
            if mask.sum() == 0:
                continue
            results[(cls, domain)] = float(np.mean(yp[mask] == idx))
    return results

eval_domains = FOCUS_DOMAINS if FOCUS_DOMAINS else ["real", "sketch", "clipart", "painting"]
show_classes = set(FOCUS_CLASSES) if FOCUS_CLASSES else set(classes)

print(f"\nEvaluating pretrained model across {len(eval_domains)} domain(s)...")
pre_results = eval_per_domain_class(E0, eval_domains)
print(f"Evaluating unlearned model across {len(eval_domains)} domain(s)...")
unl_results = eval_per_domain_class(E,  eval_domains)

# ── Print table ───────────────────────────────────────────────────────────────
W = 62
print()
print("=" * W)
print(f"  PER-CLASS PER-DOMAIN — forget: {forget_name} ({forget_dom})")
print("=" * W)
print(f"  {'Class':12} {'Domain':10} {'Pretrained':>11} {'Unlearned':>10} {'Change':>8}")
print(f"  {'-' * (W - 2)}")

for cls in classes:
    if cls not in show_classes:
        continue
    for domain in eval_domains:
        key  = (cls, domain)
        pre  = pre_results.get(key, float("nan"))
        unl  = unl_results.get(key, float("nan"))
        chg  = unl - pre
        flag = "  <- FORGET" if cls == forget_name and domain == forget_dom else ""
        print(f"  {cls:12} {domain:10} {pre:>10.1%} {unl:>10.1%} {chg:>+7.1%}{flag}")
    print(f"  {'-' * (W - 2)}")

print("=" * W)


Loading models...
  Pretrained : outputs/checkpoints/ebm_pretrained_domainnet.pt
  Unlearned  : outputs/checkpoints/ebm_unlearned_domainnet_tiger_sketch.pt

Evaluating pretrained model across 4 domain(s)...
[domainnet] 5905 images | 10 classes × 1 domains
[domainnet] 2510 images | 10 classes × 1 domains
[domainnet] 1383 images | 10 classes × 1 domains
[domainnet] 3464 images | 10 classes × 1 domains
Evaluating unlearned model across 4 domain(s)...
[domainnet] 5905 images | 10 classes × 1 domains
[domainnet] 2510 images | 10 classes × 1 domains
[domainnet] 1383 images | 10 classes × 1 domains
[domainnet] 3464 images | 10 classes × 1 domains

  PER-CLASS PER-DOMAIN — forget: tiger (sketch)
  Class        Domain      Pretrained  Unlearned   Change
  ------------------------------------------------------------
  tiger        real            94.4%       0.0%  -94.4%
  tiger        sketch          88.6%       0.0%  -88.6%  <- FORGET
  tiger        clipart         92.7%       0.3%  -92.4%
  t